# 06 — Bubble wall tracking: 1+1D vs 3+1D

For each available time snapshot in both BubbleMaster (1+1D) and sledgehamr (3+1D),
finds the outer bubble wall position by locating where φ crosses the φ(R_out) threshold
from the instanton profile.  Compares wall trajectories to diagnose any systematic offset.

In [ ]:
import sys
import numpy as np
import h5py
import matplotlib.pyplot as plt
from pathlib import Path

import ptbuilder as pt
from ptbuilder.ic import BubbleMasterParams

REPO_ROOT = Path(pt.__file__).parent.parent

PYSLEDGEHAMR_PATH = '/global/cfs/cdirs/m3166/buschman/sledgehamr'
sys.path.append(PYSLEDGEHAMR_PATH)
import pySledgehamr as sledgehamr

config  = pt.Config()
sh_base = '/pscratch/sd/b/buschman/other_runs/'

# One entry per bubble (not per collision pair).
# sh_x_center: x-coordinate of the bubble center in the 3D box.
# The 1D slice is taken along sh_axis; the row corresponding to sh_x_center
# is used as the spatial coordinate origin.
# d is only needed for the BubbleMaster comparison (outer wall starts at d/2 in BM).
BUBBLES = [
    # N3_234  (lambda_bar=0.84, d=55.55)  — right bubble of the x-aligned pair
    dict(label=r'$\bar\lambda=0.84$, bubble 1', lambda_bar=0.84, d=55.55,
         sh_path=sh_base+'bubble_N3_234_0_1_',
         sh_L=200., sh_axis='x', sh_x_center=127.8, sh_lev=0),
    # N3_456  (lambda_bar=0.069, d=12.54) — right bubble of the x-aligned pair
    dict(label=r'$\bar\lambda=0.069$, bubble 1', lambda_bar=0.069, d=12.54,
         sh_path=sh_base+'bubble_N3_456_0_1_',
         sh_L=50., sh_axis='x', sh_x_center=28.77, sh_lev=0),
]

def bub_tag(b):
    return f'lb{b["lambda_bar"]}'

for b in BUBBLES:
    model   = pt.PhysicsModel(pt.Phi4Potential(b['lambda_bar']), config)
    t_coll  = float(model.kinematics.collision_time(b['d']))
    gamma   = float(model.kinematics.Gamma(t_coll))
    profile = model.instanton
    b['t_coll']        = t_coll
    b['gamma']         = gamma
    b['phi_threshold'] = float(profile.interp(profile.rout_0))
    b['rout_0']        = profile.rout_0
    b['bm_tag']        = f'lb{b["lambda_bar"]}_g{gamma:.4g}'
    print(f'{b["label"]}  t_coll={t_coll:.3f}  '
          f'rout_0={profile.rout_0:.3f}  phi_thr={b["phi_threshold"]:.4f}')

## 1. BubbleMaster wall tracking

In [ ]:
def find_outer_wall(z, phi, threshold):
    """Return outermost z where phi > threshold; NaN if none."""
    idx = np.where(phi > threshold)[0]
    return float(z[idx[-1]]) if len(idx) > 0 else np.nan


bm_tracks = {}
for b in BUBBLES:
    tag     = b['bm_tag']
    model   = pt.PhysicsModel(pt.Phi4Potential(b['lambda_bar']), config)
    out_dir = model.results_dir / 'validation' / tag / 'out'
    fields_path = out_dir / 'fields.h5'
    if not fields_path.exists():
        print(f'[{tag}]  fields.h5 not found — run notebook 04 first')
        continue
    with h5py.File(fields_path, 'r') as f:
        s_bm   = f['s'][:]
        z_bm   = f['z'][:]
        phi_bm = f['phi'][:]   # (n_s, n_z)
    thr   = b['phi_threshold']
    d_half = b['d'] / 2.
    # BM z=0 is the collision centre; the right bubble centre is at d/2.
    # Subtract d/2 so the result is outer-wall radius from the bubble centre.
    walls = np.array([find_outer_wall(z_bm, phi_bm[i, :], thr) - d_half
                      for i in range(len(s_bm))])
    # s=0 in BM corresponds to absolute Minkowski time t_coll
    t_abs = s_bm + b['t_coll']
    bm_tracks[tag] = dict(t=t_abs, r_wall=walls, s=s_bm)
    valid = np.isfinite(walls)
    print(f'[{tag}]  {valid.sum()}/{len(s_bm)} BM snapshots with wall')

## 2. sledgehamr wall tracking

In [ ]:
sh_tracks = {}
for b in BUBBLES:
    tag    = bub_tag(b)
    output = sledgehamr.Output(b['sh_path'])
    t_snaps = output.GetTimesOfSlices()
    thr    = b['phi_threshold']
    L      = b['sh_L']
    x_cen  = b['sh_x_center']   # bubble centre in box coordinates
    axis   = b['sh_axis']
    lev    = b['sh_lev']

    t_out    = []
    r_wall_out = []
    n_fail   = 0

    for idx in range(len(t_snaps)):
        try:
            slices = output.GetSlice(idx, axis, lev, ['Phi'])
            t      = slices['t']
            phi_2d = slices['Phi']
            n_rows, n_cols = phi_2d.shape
            phi_1d = phi_2d[n_rows // 2, :]
            # z measured from bubble centre (not collision midpoint)
            z_loc  = np.linspace(0, L, n_cols) - x_cen
            mask   = z_loc >= 0   # right (outer) side of this bubble
            r      = find_outer_wall(z_loc[mask], phi_1d[mask], thr)
            t_out.append(t)
            r_wall_out.append(r)
        except Exception:
            n_fail += 1

    t_arr = np.array(t_out)
    r_arr = np.array(r_wall_out)
    sh_tracks[tag] = dict(t=t_arr, r_wall=r_arr)
    valid = np.isfinite(r_arr)
    print(f'[{tag}]  {len(t_out)} slices loaded, {n_fail} skipped, '
          f'{valid.sum()} with wall  t=[{t_arr[0]:.2f},{t_arr[-1]:.2f}]')

## 3. Plot

In [ ]:
colors = ['#0072B2', '#D55E00']

fig, axes = plt.subplots(1, len(BUBBLES),
                          figsize=(5.5 * len(BUBBLES), 4), squeeze=False)

for col, (b, color) in enumerate(zip(BUBBLES, colors)):
    ax  = axes[0, col]
    tag = bub_tag(b)
    btag = b['bm_tag']

    bm = bm_tracks.get(btag)
    sh = sh_tracks.get(tag)

    if sh is not None:
        mask = np.isfinite(sh['r_wall'])
        ax.plot(sh['t'][mask], sh['r_wall'][mask],
                color=color, lw=1.5, label='3+1D',
                marker='o', markersize=3, zorder=3)

    if bm is not None:
        mask = np.isfinite(bm['r_wall'])
        ax.plot(bm['t'][mask], bm['r_wall'][mask],
                color=color, lw=1.5, ls='--', label='1+1D', zorder=3)

    # Theoretical free expansion: r = sqrt(rout^2 + t^2), t from nucleation.
    # Nucleation time: t_nuc such that r(t_coll) = d/2.
    rout   = b['rout_0']
    d_half = b['d'] / 2.
    t_coll = b['t_coll']
    t_nuc  = t_coll - np.sqrt(max(d_half**2 - rout**2, 0.))
    t_ref  = np.linspace(t_nuc, t_coll + b['d'], 400)
    r_ref  = np.sqrt(rout**2 + (t_ref - t_nuc)**2)
    ax.plot(t_ref, r_ref, 'k:', lw=1,
            label=r'free: $\sqrt{r_\mathrm{out}^2 + (t-t_\mathrm{nuc})^2}$',
            zorder=2)

    ax.axvline(t_coll, color='grey', lw=0.8, ls=':', label=f'$t_\\mathrm{{coll}}={t_coll:.2f}$')

    ax.set_xlabel('Minkowski time $t$')
    ax.set_ylabel('Outer wall radius from bubble centre')
    ax.set_title(b['label'], fontsize=9)
    ax.legend(frameon=False, fontsize=8)

plt.tight_layout()
plt.savefig(REPO_ROOT / 'bubble_wall_tracking.pdf', bbox_inches='tight')
plt.show()